In [4]:
# Q1: K-Fold Cross Validation for Multiple Linear Regression
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# step 0: load dataset directly from URL
url = "https://drive.google.com/uc?export=download&id=1O_NwpJT-8xGfU_-3llUl2sgPu0xllOrX"
data = pd.read_csv(url)
print("Dataset loaded, first 5 rows:\n", data.head())

# split into input (X) and output (y)
X = data.drop("Price", axis=1).values
y = data["Price"].values.reshape(-1, 1)

# scale input features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 5-fold cross validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2_list = []
beta_list = []

for train_idx, test_idx in kf.split(X):
    Xtr, Xte = X[train_idx], X[test_idx]
    ytr, yte = y[train_idx], y[test_idx]

    # add bias column
    Xtr_b = np.c_[np.ones((Xtr.shape[0], 1)), Xtr]
    Xte_b = np.c_[np.ones((Xte.shape[0], 1)), Xte]

    # normal equation
    beta = np.linalg.inv(Xtr_b.T @ Xtr_b) @ (Xtr_b.T @ ytr)

    # predictions
    ypred = Xte_b @ beta
    r2 = r2_score(yte, ypred)

    r2_list.append(r2)
    beta_list.append(beta)

# best beta
best_i = np.argmax(r2_list)
best_beta = beta_list[best_i]

print("\nR² scores from 5 folds:", r2_list)
print("Best R²:", r2_list[best_i])

# train on 70%, test on 30% using best beta
X_train70, X_test30, y_train70, y_test30 = train_test_split(X, y, test_size=0.3, random_state=42)
X_train70_b = np.c_[np.ones((X_train70.shape[0], 1)), X_train70]
X_test30_b = np.c_[np.ones((X_test30.shape[0], 1)), X_test30]

y_pred30 = X_test30_b @ best_beta
print("Final R² on 30% test set:", r2_score(y_test30, y_pred30))


Dataset loaded, first 5 rows:
    Avg. Area Income  Avg. Area House Age  Avg. Area Number of Rooms  \
0       79545.45857             5.682861                   7.009188   
1       79248.64245             6.002900                   6.730821   
2       61287.06718             5.865890                   8.512727   
3       63345.24005             7.188236                   5.586729   
4       59982.19723             5.040555                   7.839388   

   Avg. Area Number of Bedrooms  Area Population         Price  
0                          4.09      23086.80050  1.059034e+06  
1                          3.09      40173.07217  1.505891e+06  
2                          5.13      36882.15940  1.058988e+06  
3                          3.26      34310.24283  1.260617e+06  
4                          4.23      26354.10947  6.309435e+05  

R² scores from 5 folds: [0.9179971706985147, 0.9145677884802819, 0.9116116385364478, 0.9193091764960817, 0.9243869413350317]
Best R²: 0.924386941335031

In [5]:
# Q2: Validation set for Gradient Descent Optimization
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# split: 56% train, 14% validation, 30% test
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train56, X_val14, y_train56, y_val14 = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

# add bias column
X_train56_b = np.c_[np.ones((X_train56.shape[0], 1)), X_train56]
X_val14_b = np.c_[np.ones((X_val14.shape[0], 1)), X_val14]
X_test_b = np.c_[np.ones((X_test.shape[0], 1)), X_test]

# try different learning rates
lr_list = [0.001, 0.01, 0.1, 1]
iters = 1000
best_lr, best_beta, best_r2 = None, None, -np.inf

for lr in lr_list:
    beta = np.zeros((X_train56_b.shape[1], 1))

    # gradient descent loop
    for i in range(iters):
        grad = (2 / X_train56_b.shape[0]) * (X_train56_b.T @ (X_train56_b @ beta - y_train56))
        beta = beta - lr * grad

    # validation performance
    y_val_pred = X_val14_b @ beta
    r2val = r2_score(y_val14, y_val_pred)
    print(f"LR={lr} → Validation R²={r2val:.4f}")

    if r2val > best_r2:
        best_lr, best_beta, best_r2 = lr, beta, r2val

print("\nBest learning rate:", best_lr, "with Validation R²:", best_r2)

# test performance with best beta
y_test_pred = X_test_b @ best_beta
print("Final Test R²:", r2_score(y_test, y_test_pred))


LR=0.001 → Validation R²=0.6820
LR=0.01 → Validation R²=0.9098
LR=0.1 → Validation R²=0.9098
LR=1 → Validation R²=-inf

Best learning rate: 0.01 with Validation R²: 0.909799626728122
Final Test R²: 0.9147569598865972


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1275: RuntimeWarning: overflow encountered in square
  numerator = xp.sum(weight * (y_true - y_pred) ** 2, axis=0)
